# Emerging Technologies

In [1]:
import numpy as np
import random
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import Aer
import matplotlib.pyplot as plt


## Problem 1: Generating Random Boolean Functions
**Deutsch's** algorithm is used along with another quantum algorithm to achieve faster computation. It is used to determine if a function is either balanced or constant, needing only one query. The amount of either $\textit{0}$ or $\textit{1}$ bits outputted determines whether its balanced or constant.

**Deutsch-Jozsa** algorithm came about when Deutsch and Jozsa worked further on the algorithm, that was initially created by Deutsch, into more qubits (quantum bits). This allowed for more computational power for quantum computers. It works similarly to *Deutsch's* algorithm, determining whether a function is balanced or constant. However, this algorithm goes from $\textit{n}$ bits to $\textit{1}$:
- Balanced = $\textit{0}$, $\textit{1}$ outputs equal amount of times
- Constant = Always outputs $\textit{0}$ or $\textit{1}$

### Helper Function

In [2]:
def bin_args_to_int(*args):
    """Change an arbitrary number of binary arguments to an integer value.

    Accepts 0 and 1, and True or False values:
        Returns corresponding integer
    """
    return int(''.join('1' if i else '0' for i in args), 2)

The process goes as follows:
1. **Declare $\textit{n}$ value.**
    - $\textit{n}$ = 4
    - Statically assigned as the brief defines the function to only ever take four arguments.

2. **Randomly assign `constant` or `balanced` for function type.**
    - `random` [(see official documentation)](https://docs.python.org/3/library/random.html) is used to ensure that there is unpredicablity in the generation of Boolean values.
    - `.choice()` chooses a randomly selected value from True/False array. 

3. **Build truth table for the random function.**
    - If the function type `ftype` is `constant`:
        - `.choice()`[(see official documentation)](https://docs.python.org/3/library/random.html#random.choice) randomly selects $\textit{True}$ or $\textit{False}$.
        - Fills entire table ($2^n$) with selected value. 
    - If the function type is `balanced`:
        - Table is filled with half $\textit{True}$ and $\textit{False}$ values.
        - `.shuffle()` [(see official documentation)](https://docs.python.org/3/library/random.html#random.shuffle) randomises the order of the Boolean values.
            - Ensures unpredicability.

In [3]:
def random_constant_balanced():
    """
    Generates a random Boolean function that is constant or balanced.

    The function takes in four Boolean arguments and returns True or False.
        Constant = Always returns False or always True
        Balanced = Returns True for exactly half of the inputs

    Returns:
        Function that takes four Boolean arguments
        Boolean value and type of function randomly selected
    """
    n = 4 # Fixed as specified by Problem criteria
    ftype = random.choice(['constant', 'balanced'])

    # Checks if randomly selected function is either constant or balanced
    # Build the truth table of 2^n output values (2^4 = 16 outputs)
    if ftype == 'constant':
        value = random.choice([False, True])
        table = [value] * 2**n
    # If balanced
    # Generate exactly half False and True in a random order
    else:
        table = [False] * (2**(n-1)) + [True] * (2**(n-1))
        random.shuffle(table)

    # Closure function
    def f(a, b, c, d):
        """Return True or False depending on the input."""
        return table[bin_args_to_int(a, b, c, d)]

    return f

In [4]:
# Generate a random constant or balanced function
random_function = random_constant_balanced()

# Print out results from randomly selected function
print(f"{'a':>2} {'b':>2} {'c':>2} {'d':>2}  ->  output")
print("-" * 28)

for i in range(16):
    a, b, c, d = (i >> 3) & 1, (i >> 2) & 1, (i >> 1) & 1, i & 1
    out = random_function(a, b, c, d)
    print(f"{a:>2} {b:>2} {c:>2} {d:>2}  ->  {out}")


 a  b  c  d  ->  output
----------------------------
 0  0  0  0  ->  False
 0  0  0  1  ->  True
 0  0  1  0  ->  True
 0  0  1  1  ->  True
 0  1  0  0  ->  True
 0  1  0  1  ->  True
 0  1  1  0  ->  False
 0  1  1  1  ->  True
 1  0  0  0  ->  False
 1  0  0  1  ->  False
 1  0  1  0  ->  False
 1  0  1  1  ->  False
 1  1  0  0  ->  False
 1  1  0  1  ->  False
 1  1  1  0  ->  True
 1  1  1  1  ->  True


## Problem 2: Classical Testing for Function Type

In [5]:
def determine_constant_balanced(f):
    """
    Checks if a Boolean function with four inputs is constant or balanced, using function f from Problem 1 as input.
    
    Returns:
        "constant" if function is constant, with the same values for all of the inputs
        "balanced" if function is balanced, with different values for half of the inputs
    """
    # Calculate result for first input
    first_input = f(0, 0, 0, 0)

    for i in range(1, 16):
        # Extract each bit of i using right shift, removing the least significant bit for each variable
        a = (i >> 3) & 1
        b = (i >> 2) & 1
        c = (i >> 1) & 1
        d = i & 1

        if f(a, b, c, d) != first_input:
            return "balanced"
        
    # If all results match, it's constant
    return "constant"

In [6]:
# Testing for Problem 2
# Test randomly generated functions, checking the result against all outputs
print("Testing Problem 2         | Status:")
for i in range(10):
    test_function = random_constant_balanced()
    result = determine_constant_balanced(test_function)

    # Get all outputs and check if they all match
    outputs = [test_function((j >> 3) & 1, (j >> 2) & 1, (j >> 1) & 1, j & 1) for j in range(16)]
    expected = "constant" if len(set(outputs)) == 1 else "balanced"

    status = "Passed" if result == expected else "Failed"
    print(f"  Function {i + 1:>2}: {result:<10} | {status}")


Testing Problem 2         | Status:
  Function  1: balanced   | Passed
  Function  2: constant   | Passed
  Function  3: constant   | Passed
  Function  4: balanced   | Passed
  Function  5: constant   | Passed
  Function  6: constant   | Passed
  Function  7: constant   | Passed
  Function  8: constant   | Passed
  Function  9: balanced   | Passed
  Function 10: constant   | Passed


## Problem 3: Quantum Oracles

In [7]:
def oracle_constant_0():
    qc = QuantumCircuit(2)

    return qc

In [8]:
def oracle_constant_1():
    qc = QuantumCircuit(2)
    qc.x(1)

    return qc

In [ ]:
def oracle_identity():
    qc = QuantumCircuit(2)
    qc.cx(0, 1)
    
    return qc

In [ ]:
def oracle_not():
    qc = QuantumCircuit(2)
    qc.x(1)
    qc.cx(0, 1)

    return qc

### Display Oracles

In [ ]:
oracle_labels = {
    "f(x) = 0": oracle_constant_0,
    "f(x) = 1": oracle_constant_1,
    "f(x) = x": oracle_identity,
    "f(x) = ¬x": oracle_not
}


## Problem 4: Deutsch's Algorithm with Qiskit

## Problem 5: Scaling to the Deutsch-Jozsa Algorithm